# 02 — Training, optimisation and evaluation

## 1. Purpose

**What:** train, optimise and evaluate the sentiment models (negative / neutral / positive)
on the cleaned Financial PhraseBank data. This notebook is extended step by step: each
modelling iteration gets its own section and one row in `outputs/metrics/iterations.csv`.

**Why:** a fixed, documented protocol makes the iterations comparable and the final result
trustworthy.

> **Integrity rule:** the 20 % test set stays untouched until the final evaluation.
> All model choices (features, hyperparameters, model type) are made with cross-validation
> on the training part only. The test set is evaluated exactly once, for the final model.

In [1]:
# Imports and shared settings
import pandas as pd
from sklearn.dummy import DummyClassifier

from newsentiment import config
from newsentiment.data import load_training_data
from newsentiment.evaluation import (
    create_split,
    cross_val_predictions,
    cross_validate_model,
    evaluate,
    load_split,
    log_iteration,
)
from newsentiment.model import build_pipeline, key_params

LABELS = list(config.LABELS)  # fixed class order: negative, neutral, positive

## 2. Data and split

**What:** load the cleaned data, create the stratified 80/20 split and apply it.
`create_split()` writes `data/split.json` (row positions plus a fingerprint of the data) on the
first run and simply reuses the file afterwards; `load_split()` refuses to apply it if the data
has changed.

**Why:** a stored split guarantees that every iteration, and the final evaluation, sees exactly
the same training and test rows. Stratification keeps the class shares equal in both parts, which
matters because the classes are imbalanced.

In [2]:
# Cleaned training data (see 01_eda.ipynb for the cleaning steps)
df, report = load_training_data()

# Create the split on the first run, reuse data/split.json afterwards
split = create_split(df)
train_df, test_df = load_split(df)

print(f"Rows: {len(df):,} | split file: {config.SPLIT_FILE.relative_to(config.PROJECT_ROOT).as_posix()}")

Rows: 4,836 | split file: data/split.json


In [3]:
# Size and class counts per part; the shares should be (almost) identical
parts = {"full": df, "train": train_df, "test": test_df}
split_table = pd.DataFrame(
    {name: part["label"].value_counts().reindex(LABELS) for name, part in parts.items()}
).T
split_table["total"] = split_table.sum(axis=1)
for label in LABELS:
    split_table[f"{label}_%"] = (split_table[label] / split_table["total"] * 100).round(1)
split_table

label,negative,neutral,positive,total,negative_%,neutral_%,positive_%
full,604,2871,1361,4836,12.5,59.4,28.1
train,483,2296,1089,3868,12.5,59.4,28.2
test,121,575,272,968,12.5,59.4,28.1


## 3. Evaluation protocol

**What:** every candidate model is scored with **5-fold stratified cross-validation on the
training part** (shuffled, fixed seed, so all models see identical folds). We report mean ± standard
deviation over the folds.

**Main metric: macro-F1** — the unweighted mean of the per-class F1 scores. With about 59 %
neutral sentences, accuracy rewards a model that mostly predicts "neutral"; macro-F1 gives the
small negative class the same weight as the others. Accuracy is reported as a secondary metric.
For the final model, per-class precision/recall and the confusion matrix are added
(`evaluate()` in `newsentiment.evaluation`).

## 4. Iteration "baseline": majority class

**What:** a `DummyClassifier` that always predicts the most frequent class (neutral), ignoring
the text.

**Why:** it is the lower bound every real model must clearly beat. Its accuracy equals the
neutral share, while its macro-F1 is low (F1 = 0 for negative and positive), which shows why
accuracy alone would be misleading.

In [4]:
# Cross-validate the majority-class baseline on the training part only
baseline = DummyClassifier(strategy="most_frequent")
cv_baseline = cross_validate_model(baseline, train_df["text"], train_df["label"])

print(f"Accuracy: {cv_baseline['accuracy_mean']:.3f} ± {cv_baseline['accuracy_std']:.3f}")
print(f"Macro-F1: {cv_baseline['macro_f1_mean']:.3f} ± {cv_baseline['macro_f1_std']:.3f}")

# Per-fold scores
pd.DataFrame(
    {"accuracy": cv_baseline["fold_accuracy"], "macro_f1": cv_baseline["fold_macro_f1"]},
    index=pd.RangeIndex(1, config.CV_FOLDS + 1, name="fold"),
).round(4)

Accuracy: 0.594 ± 0.000
Macro-F1: 0.248 ± 0.000


,accuracy,macro_f1
fold,,
1,0.5943,0.2485
2,0.5930,0.2482
3,0.5930,0.2482
4,0.5938,0.2484
5,0.5938,0.2484


In [5]:
# Log the iteration (re-running replaces the existing "baseline" row)
iterations = log_iteration(
    "baseline",
    "Majority class",
    "DummyClassifier(strategy='most_frequent'); lower bound for all later models",
    cv_baseline,
    baseline.get_params(),
)
iterations

,iteration,model,description,cv_accuracy_mean,cv_accuracy_std,cv_macro_f1_mean,cv_macro_f1_std,params,logged_at
0,baseline,Majority class,DummyClassifier(strategy='most_frequent'); low...,0.5936,0.0005,0.2483,0.0001,"{""constant"": null, ""random_state"": null, ""stra...",2026-11-10T12:45:31
1,v0,TF-IDF + Multinomial NB,"TF-IDF unigrams (defaults, no stop-word remova...",0.6657,0.0126,0.4116,0.0218,"{""classifier"": ""MultinomialNB"", ""clf__alpha"": ...",2026-11-10T12:45:01
2,v1a,TF-IDF + logistic regression,TF-IDF unigrams (defaults) + LogisticRegressio...,0.7482,0.0167,0.6443,0.0278,"{""classifier"": ""LogisticRegression"", ""clf__C"":...",2026-11-10T12:45:03
3,v1b,TF-IDF + logistic regression (balanced),"As v1a, with class_weight='balanced' to counte...",0.7549,0.0106,0.7100,0.0135,"{""classifier"": ""LogisticRegression"", ""clf__C"":...",2026-11-10T12:45:04


## 5. Iteration v0: TF-IDF + Multinomial Naive Bayes

**What:** the first text model. `TfidfVectorizer` with default settings (unigrams, lowercase,
**no stop-word removal**) turns each sentence into a weighted bag of words; Multinomial Naive Bayes
(default `alpha=1.0`) classifies it.

**Why:** Naive Bayes is the classic fast baseline for text classification. It shows how much a
plain bag of words already gains over the majority baseline. Stop words are kept on purpose:
scikit-learn's list contains *up, down, less, more, not, no*, which carry sentiment in financial news.
The vectorizer is part of the pipeline, so it is refitted inside every fold and never sees the
held-out sentences.

In [6]:
# Training inputs and labels (the test part is not used here)
X_train, y_train = train_df["text"], train_df["label"]

# v0: TF-IDF (unigrams, defaults) + Multinomial Naive Bayes
v0 = build_pipeline("nb")
cv_v0 = cross_validate_model(v0, X_train, y_train)

print(f"Accuracy: {cv_v0['accuracy_mean']:.3f} ± {cv_v0['accuracy_std']:.3f}")
print(f"Macro-F1: {cv_v0['macro_f1_mean']:.3f} ± {cv_v0['macro_f1_std']:.3f}")

# Log the iteration (re-running replaces the existing "v0" row)
log_iteration(
    "v0",
    "TF-IDF + Multinomial NB",
    "TF-IDF unigrams (defaults, no stop-word removal) + MultinomialNB(alpha=1.0)",
    cv_v0,
    key_params(v0),
);

Accuracy: 0.666 ± 0.013
Macro-F1: 0.412 ± 0.022


## 6. Iteration v1: TF-IDF + logistic regression

**What:** the same TF-IDF features with logistic regression, the main model of the concept.
- **v1a:** no class weights (every sentence counts equally).
- **v1b:** `class_weight="balanced"`: errors on a class are weighted inversely to its frequency,
  so the small negative class counts as much as the large neutral class.

**Why:** logistic regression learns the word weights discriminatively and, unlike Naive Bayes,
does not assume that words are independent. Class weights address the imbalance (about 12 %
negative sentences) directly in the loss; comparing v1a and v1b isolates their effect.

In [7]:
# v1a: logistic regression without class weights
v1a = build_pipeline("logreg")
cv_v1a = cross_validate_model(v1a, X_train, y_train)
log_iteration(
    "v1a",
    "TF-IDF + logistic regression",
    "TF-IDF unigrams (defaults) + LogisticRegression(C=1.0), no class weights",
    cv_v1a,
    key_params(v1a),
)

# v1b: same model, errors weighted inversely to class frequency
v1b = build_pipeline("logreg", clf__class_weight="balanced")
cv_v1b = cross_validate_model(v1b, X_train, y_train)
log_iteration(
    "v1b",
    "TF-IDF + logistic regression (balanced)",
    "As v1a, with class_weight='balanced' to counter the class imbalance",
    cv_v1b,
    key_params(v1b),
)

for name, cv in [("v1a", cv_v1a), ("v1b", cv_v1b)]:
    print(
        f"{name}: accuracy {cv['accuracy_mean']:.3f} ± {cv['accuracy_std']:.3f} | "
        f"macro-F1 {cv['macro_f1_mean']:.3f} ± {cv['macro_f1_std']:.3f}"
    )

v1a: accuracy 0.748 ± 0.017 | macro-F1 0.644 ± 0.028
v1b: accuracy 0.755 ± 0.011 | macro-F1 0.710 ± 0.013


## 7. Comparison

**What:** (a) the cross-validation scores of all iterations so far, read back from
`iterations.csv`; (b) per-class precision, recall and F1 from **out-of-fold predictions**:
`cross_val_predictions()` uses the same five folds, and every prediction comes from a model that
did not see that sentence. The per-class table is saved as `outputs/metrics/cv_per_class.csv`.

**Why:** the mean macro-F1 decides which model is best; the per-class view shows *where* the
models differ, especially on the small negative class. Both come from the training part only.

In [8]:
# (a) CV scores of all iterations, as logged
log = pd.read_csv(config.ITERATIONS_FILE)
comparison = pd.DataFrame(
    {
        "model": log["model"],
        "CV accuracy": [f"{m:.3f} ± {s:.3f}" for m, s in zip(log["cv_accuracy_mean"], log["cv_accuracy_std"])],
        "CV macro-F1": [f"{m:.3f} ± {s:.3f}" for m, s in zip(log["cv_macro_f1_mean"], log["cv_macro_f1_std"])],
    }
)
comparison.index = log["iteration"]
comparison

,model,CV accuracy,CV macro-F1
iteration,,,
baseline,Majority class,0.594 ± 0.001,0.248 ± 0.000
v0,TF-IDF + Multinomial NB,0.666 ± 0.013,0.412 ± 0.022
v1a,TF-IDF + logistic regression,0.748 ± 0.017,0.644 ± 0.028
v1b,TF-IDF + logistic regression (balanced),0.755 ± 0.011,0.710 ± 0.013


In [9]:
# (b) Per-class metrics from out-of-fold predictions (same folds as above)
models = {"v0": v0, "v1a": v1a, "v1b": v1b}
rows = []
for name, model in models.items():
    oof_pred = cross_val_predictions(model, X_train, y_train)
    result = evaluate(y_train, oof_pred)
    for label in LABELS:
        rows.append({"iteration": name, "class": label, **result["per_class"][label]})

per_class = pd.DataFrame(rows, columns=["iteration", "class", "precision", "recall", "f1", "support"])
per_class_file = config.METRICS_DIR / "cv_per_class.csv"
per_class.round(4).to_csv(per_class_file, index=False)

# Wide view: one row per iteration, metrics per class side by side
per_class.pivot(index="iteration", columns="class", values=["precision", "recall", "f1"]).round(3)

precision                    recall                        f1  \
class      negative neutral positive negative neutral positive negative   
iteration                                                                 
v0            0.929   0.671    0.623    0.027   0.983    0.281    0.052   
v1a           0.823   0.743    0.747    0.366   0.949    0.493    0.507   
v1b           0.641   0.825    0.659    0.654   0.822    0.657    0.648   

                            
class     neutral positive  
iteration                   
v0          0.797    0.387  
v1a         0.834    0.594  
v1b         0.824    0.658

## 8. Observations

Measured with 5-fold cross-validation on the training part (3,868 sentences); the test set was not used.

- **All three text models clearly beat the majority baseline** (macro-F1 0.248):
  v0 (Naive Bayes) 0.412 ± 0.022, v1a (logistic regression) 0.644 ± 0.028,
  v1b (logistic regression, balanced) 0.710 ± 0.013.
- **Best model by macro-F1: v1b** (0.710), also the highest accuracy (0.755 ± 0.011) and the
  smallest fold-to-fold spread of the three text models.
- **Naive Bayes with defaults almost never predicts "negative":** out-of-fold recall 0.027
  (13 of 483 negative sentences), with neutral recall 0.983. Its accuracy (0.666) is only
  7 points above the baseline.
- **Effect of class weights on negative recall:** 0.366 (v1a) → 0.654 (v1b); negative F1
  0.507 → 0.648. Positive recall rises as well (0.493 → 0.657). The cost is lower negative
  precision (0.823 → 0.641) and lower neutral recall (0.949 → 0.822).

v1b is the starting point for the next iterations (n-grams and hyperparameter tuning).